# Implement Random Forest Algorithm from Scratch

In [1]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

### Data Analysis

In [ ]:
iris = pd.read_csv("../../../data/Iris.csv")

In [8]:
iris.drop('Id', inplace = True, axis = 1)

In [9]:
iris.head()

,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,5.1,3.5,1.4,0.2,Iris-setosa
1,4.9,3.0,1.4,0.2,Iris-setosa
2,4.7,3.2,1.3,0.2,Iris-setosa
3,4.6,3.1,1.5,0.2,Iris-setosa
4,5.0,3.6,1.4,0.2,Iris-setosa


In [10]:
iris.head().style.background_gradient(cmap=sns.light_palette("seagreen", as_cmap=True))

,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,5.100000,3.500000,1.400000,0.200000,Iris-setosa
1,4.900000,3.000000,1.400000,0.200000,Iris-setosa
2,4.700000,3.200000,1.300000,0.200000,Iris-setosa
3,4.600000,3.100000,1.500000,0.200000,Iris-setosa
4,5.000000,3.600000,1.400000,0.200000,Iris-setosa


In [11]:
X_df = iris.iloc[:, :-1] # Set our training dataframe

y_df = iris.iloc[:, -1] # Set the training lables dataframe

In [12]:
fig = px.pie(iris, 'Species',color_discrete_sequence=['#3dec84 ','#009688 ','#2E8B57 '],title='Data Distribution',template='plotly')

fig.show()

It can be easily identified that the labels are perfectly balanced. 

### Sepal-Length

In [13]:
fig = px.box(data_frame=iris, x='Species',y='SepalLengthCm',color='Species',color_discrete_sequence=['#3dec84 ','#009688 ','#2E8B57 '],orientation='v')
fig.show()

In [14]:
fig = px.histogram(data_frame=iris, x='SepalLengthCm',color='Species',color_discrete_sequence=['#3dec84 ','#009688 ','#2E8B57 '],nbins=50)
fig.show()

From these plots we conclude that:

- Setosa has much smaller SepalLength than the other 2 classes

- Virginca has the highest SepalLength, however It seems hard to distingush between Virginca and Versicolor using SepalLength as the difference is less clear

- We can see that Virginica contains an outlier

### Sepal Width

In [15]:
fig = px.box(data_frame=iris, x='Species',y='SepalWidthCm',color='Species',color_discrete_sequence=['#3dec84 ','#009688 ','#2E8B57 '],orientation='v')
fig.show()

In [16]:
fig = px.histogram(data_frame=iris, x='SepalWidthCm',color='Species',color_discrete_sequence=['#3dec84 ','#009688 ','#2E8B57 '],nbins=30)
fig.show()

From these plots we conclude that:

- Setosa has larger SepalWidth than the other 2 classes

- Versicolo has smaller SepalWidth than the other 2 classes

- Overall all classes seem to have relatively close value of sepalwidth which indicate that is might not be a very useful feature

### Petal-Length

In [17]:
fig = px.box(data_frame=iris, x='Species',y='PetalLengthCm',color='Species',color_discrete_sequence=['#3dec84 ','#009688 ','#2E8B57 '],orientation='v')
fig.show()

In [18]:
fig = px.histogram(data_frame=iris, x='PetalLengthCm',color='Species',color_discrete_sequence=['#3dec84 ','#009688 ','#2E8B57 '],nbins=30)
fig.show()

From these plots we conclude that:

- Setosa has much smaller PetaLength than the other 2 classes

- This difference is less clear between Virginica and Versicolor

- Overall this seems like an PetaLength interesting feature

### Petal-Width

In [19]:
fig = px.box(data_frame=iris, x='Species',y='PetalWidthCm',color='Species',color_discrete_sequence=['#3dec84 ','#009688 ','#2E8B57 '],orientation='v')
fig.show()

In [20]:
fig = px.histogram(data_frame=iris, x='PetalWidthCm',color='Species',color_discrete_sequence=['#3dec84 ','#009688 ','#2E8B57 '],nbins=30)
fig.show()

From these plots we conclude that:

- Setosa has much smaller PetalWidth than the other 2 classes

- This difference is less clear between Virginica and Versicolor

- Overall this seems like an PetalWidth interesting feature

In [21]:
fig = px.scatter(data_frame=iris, x='SepalLengthCm',y='SepalWidthCm'
           ,color='Species',size='PetalLengthCm',template='seaborn',color_discrete_sequence=['#3dec84 ','#009688 ','#2E8B57 '],)

fig.update_layout(width=800, height=600,
                  xaxis=dict(color="#36FF00"),
                 yaxis=dict(color="#36FF00"))
fig.show()

In [22]:
fig = px.scatter(data_frame=iris, x='PetalLengthCm',y='PetalWidthCm'
           ,color='Species',size='SepalLengthCm',template='seaborn',color_discrete_sequence=['#3dec84 ','#009688 ','#2E8B57 '],)

fig.update_layout(width=800, height=600,
                  xaxis=dict(color="#36FF00"),
                 yaxis=dict(color="#36FF00"))
fig.show()

### Data Preprocessing

In [25]:
iris['Species'] = iris['Species'].astype('category')
# `category` is a specific data type in pandas that is used to represent categorical variables.

codes = iris['Species'].cat.codes

In [27]:
iris.head()
codes

0      0
1      0
2      0
3      0
4      0
      ..
145    2
146    2
147    2
148    2
149    2
Length: 150, dtype: int8

In [28]:
def train_test_split(X, y, random_state= 42, test_size= 0.2):
    n_samples = X.shape[0]

    np.random.seed(random_state)

    shuffled_indices = np.random.permutation(np.arange(n_samples))
    
    test_size = int(test_size * n_samples)

    test_indices = shuffled_indices[:test_size]
    train_indices = shuffled_indices[test_size:]

    X_train, X_test = X[train_indices], X[test_indices]
    y_train, y_test = y[train_indices], y[test_indices]

    return X_train, X_test, y_train, y_test

In [31]:
X = iris.iloc[:, :-1].values
y = iris.iloc[:, -1].values.reshape(-1, 1)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state=41)

### Model Implementation

### Random Forest
#### How the algorithm works
For specific number of times we will:

Create training sets of the same size using sampling with replacement

Train a decision tree on the sampled data

Get the votes of the label from each tree and pick the majority vote

RandomForests are called random becuase each tree is constructed on a bootstrapped subset of our data. 

In [32]:
from sklearn.tree import DecisionTreeClassifier
m = DecisionTreeClassifier()

In [48]:
class RandomForest:
    """
    A random forest classifier.

    Parameters:
    -----------
    n_trees : int, default = 7
        The number of trees in the random forest.
    max_depth: int, default = 7
        The maximum depth of each decision tree in the random forest.
    min_samples: int, default = 2
        The minimum number of samples required to split an internal node of each decision tree in the random forest.
    
    Attributes:
    trees: list of DecisionTreeClassifier
        The decision trees in the random forest.
    """

    def __init__(self, n_trees= 7, max_depth=7, min_samples=2):
        self.n_trees = n_trees
        self.max_depth = max_depth
        self.min_samples = min_samples
        self.trees = []

    def fit(self, X, y):
        # Create an empty list to store the trees.
        self.trees = []

        # Concatenate X and y into a single dataset.
        dataset = np.concatenate((X, y.reshape(-1, 1)), axis = 1)

        # Loop over the number of trees.
        for _ in range(self.n_trees):
            # Create a decision tree instance.
            tree = DecisionTreeClassifier(max_depth=self.max_depth, min_samples_split=self.min_samples)

            # Sample from the dataset with replacement (bootstrapping)
            dataset_sample = self.bootstrap_samples(dataset)
            X_sample, y_sample = dataset_sample[:, :-1], dataset_sample[:, -1]

            tree.fit(X_sample, y_sample)

            self.trees.append(tree)
        return self

    def bootstrap_samples(self, dataset):
        """
        Bootstrap the dataset by sampling from it with replacement

        Parameters:
        ------------
        dataset: array-like of shape (n_samples, n_features + 1). The dataset to bootstrap.

        Returns:
        -----------
        dataset_sample: array-like of shape (n_samples, n_features + 1). The bootstrap dataset sample.
        """
        # Get the number of samples in the dataset.
        n_samples = dataset.shape[0]
        # Generate random indices to index into the dataset with replacement.
        np.random.seed(1)
        indices = np.random.choice(n_samples, n_samples, replace=True)
        dataset_sample = dataset[indices]
        return dataset_sample

    def most_common_label(self, y):
        """
        Return the most common label in an array of labels.

        Parameters:
        y: array-like of shape (n_samples, )
            The array of labels.
        
        Returns:
        -------
        most_occuring_value: int or flaot
            The most common label in the array.
        """
        y = list(y)
        # get the highest present class in the array
        most_occuring_value = max(y, key=y.count)
        return most_occuring_value
    
    def predict(self, X):
        """
        Predict class for X.

        Parameters
        --------
        X: array-like of shape (n_sampels, n_features)
            The input samples

        Returns
        ----------
        majority_predictions: array-like of shape (n_samples,) The predicted class
        """
        # get prediction from each tree in the tree list on the test data
        predictions = np.array([tree.predict(X) for tree in self.trees])

        # get prediction for the same sample from all trees for each sample in the test data
        preds = np.swapaxes(predictions, 0, 1) 
        # We are swapping since we are trying to get the mx vote for every input, think of it this way
        # [[1,2, 3],
        #  [3, 4, 4]]
        # each row at the moment is the prediction for every input, we want to get the max vote for every colum, thus we need to swap the first and second dimension  to
        # [[1, 3],
        #  [2, 4],
        #  [3, 4]]
        # then we take every row and take the max for every row.
        
        # get the most voted value by the trees and store it in the final predictions array
        majority_predictions = np.array([self.most_common_label(pred) for pred in preds])
        return majority_predictions
    

In [49]:
# Testting space for playing around

# Using np.random.choice
ex = np.random.choice(3, 3, replace=True)
a = np.array([[1,2, 3], 
              [4, 5, 6]])

# using np.swapaxes(dataset, dim1, dim2) allows you to swap the values in dim1 and dim2
b = np.swapaxes(a, 0, 1)
print(a)
print(b)



[[1 2 3]
 [4 5 6]]
[[1 4]
 [2 5]
 [3 6]]


### Evaluation

In [70]:
def accuracy(y_true, y_pred):
    """
        Compute the accuracy of the classification model.

        Parameters:
            y_true (numpy_array): A numpy array of trye labels for each data point.
            y_pred (numpy_array): A numpy array of predicted labels for each data point.
        
        Returns:
            float: The accuracy of the model, expressed as percentage.
    """
    y_true = np.array(y_true).flatten()
    y_pred = np.array(y_pred).flatten()
    total_samples = len(y_true)
    correct_predictions = np.sum(y_true == y_pred)
    return (correct_predictions/ total_samples)


In [71]:
y_test

[['Iris-virginica'], ['Iris-virginica'], ['Iris-virginica'], ['Iris-versicolor'], ['Iris-virginica'], ..., ['Iris-versicolor'], ['Iris-versicolor'], ['Iris-versicolor'], ['Iris-setosa'], ['Iris-versicolor']]
Length: 30
Categories (3, object): ['Iris-setosa', 'Iris-versicolor', 'Iris-virginica']

In [72]:
model = RandomForest(10, 10, 2)
model.fit(X_train, y_train)

predictions = model.predict(X_test) # evaluate the model on the test data

# print(predictions)
accuracy(y_test, predictions)

np.float64(0.9333333333333333)

In [73]:
dt = DecisionTreeClassifier()
dt.fit(X_train, y_train)
predictions = dt.predict(X_test)

accuracy(y_test, predictions)

np.float64(0.9)